In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import fnmatch
from connect import bob
from utils import *
from limb_fitting import *
from fit_cld import *
from scipy.ndimage import gaussian_filter

In [2]:
def reflection_point_predict(header):
    px = [1.63114715e-06, 6.72511045e-03, 9.60448053e+02]
    py = [4.61830880e-06, -6.85005911e-03, 9.77508840e+02]

    r_sun = header['RSUN_ARC']
    dx, dy = header['PXBEG2'] - 1, header['PXBEG1'] - 1

    xr = np.polyval(px, r_sun) - dx
    yr = np.polyval(py, r_sun) - dy
    return xr, yr


def roll(image, dx, dy):
    nx, ny = image.shape
    image_ = np.zeros_like(image)
    x, y = int(round(dx)), int(round(dy))
    image_[max(x, 0): min(nx + x, nx), max(y, 0): min(ny + y, ny)] = image[max(-x, 0): min(nx - x, nx),
                                                                     max(-y, 0): min(ny - y, ny)]
    return image_

def reflect(image, xr, yr):
    nx, ny = image.shape
    return roll(image[::-1, ::-1], 2 * int(round(xr)) - nx + 1, 2 * int(round(yr)) - ny + 1)

In [4]:
sftp = bob()

top_dir = '/data/slam/valori/test_l2_fmdb/FDT_test_release_v08_2025/v4/l2/'
#top_dir = '/data/solo/phi/data/fmdb/l1/'
dirs = sorted(sftp.listdir(top_dir))

Q = []

for directory in dirs:
    if fnmatch.fnmatch(directory, '2024-05*'):
        for file in sorted(sftp.listdir(top_dir + directory)):
            #if fnmatch.fnmatch(file, '*fdt-[ia]lam*.fits.gz'):
            if fnmatch.fnmatch(file, '*stokes*.fits.gz'):
                try:
                    print(file)

                    remote_file = top_dir + directory + '/' + file
                    local_file = 'temp.fits.gz'
                    sftp.get(remote_file, local_file)
                except:
                    pass

                stop

solo_L2_phi-fdt-stokes_20240501T031503_V202602220014_0445010501.fits.gz


NameError: name 'stop' is not defined

In [13]:
s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz')
xd, yd = s['xd'], s['yd']

with fits.open('temp.fits.gz') as hdul:
    header = hdul[0].header
    data = hdul[0].data

cpos = header['CONTPOS'] - 1
xr, yr = reflection_point_predict(header)

image = data[cpos,0].copy()
ghost = gaussian_filter(reflect(image, xr, yr), 8)
image = image - ghost * 0.01

image = undistort(image, header, xd, yd)

nx, ny = image.shape
xc, yc, rsun = find_center(image)
print(rsun)

455.0355556891086


In [6]:
xi, yi = np.mgrid[:nx, :ny]
r2 = (xi - xc) ** 2 + (yi - yc) ** 2
Q0 = neckel(np.sqrt((1 - r2 / (rsun + 0.2) ** 2).clip(0)))

In [7]:
plt.figure(figsize=(10,10))
plt.imshow(image - Q0, 'inferno', vmin=-0.01, vmax=0.01)

plt.tight_layout()

In [14]:
xi, yi = np.mgrid[:nx, :ny]
ri = np.sqrt((xi - xc) ** 2 + (yi - yc) ** 2)
r = np.arange(-0.5, np.floor(rsun + 50), 1)

q = []
for a, b in zip(r[:-1], r[1:]):
    t = np.where(np.all([ri > a, ri < b, (xi - xc) * (xc - xr) > -(yi - yc) * (yc - yr)], axis=0))
    q += [np.nanmedian(image[t])]

q = np.array(q)
q /= np.nanpercentile(q, 99)
r = (r[:-1] + r[1:]) / 2

In [19]:
plt.figure(figsize=(10,8))
plt.plot(r, q)

plt.xlim(rsun + 0.2,rsun + 0.2+50)
plt.ylim(0,0.2)
plt.tight_layout()

In [35]:
from scipy.ndimage import gaussian_filter

sigma = 0.9
alpha = 2
beta = 1.4
epsilon = 0.2

rmax = int(np.max(r)) + 1
dr = 0.25

xi, yi = np.mgrid[-rmax:rmax:dr, -rmax:rmax:dr]
r2 = xi ** 2 + yi ** 2

Q = neckel(np.sqrt((1 - r2 / (rsun + 0.2) ** 2).clip(0)))
Q = gaussian_filter(Q, sigma / dr)
Q = Q[::4,::4]


def model(r, beta=1.5, epsilon=0.25, scale=1, bias=0):
    from scipy.signal import fftconvolve
    xi, yi = np.mgrid[-255:256,-255:256]
    r2 = xi ** 2 + yi ** 2
    P = 1 / (1 + r2 / alpha ** 2) ** beta
    P /= np.sum(P)

    Q_ = fftconvolve(Q, P, mode='same')
    Q_ = Q_ * epsilon + Q * (1 - epsilon)

    q = Q_[rmax, rmax:]
    return q * scale + bias


In [36]:
from scipy.optimize import curve_fit

params, _ = curve_fit(model, r, q, bounds=([1, 0.1, 0.98, -5e-3], [2, 0.9, 1.02, 5e-3]),
                      nan_policy='omit', sigma=np.sqrt(q))
beta, epsilon, scale, bias = params
params

array([1.49482595, 0.27129528, 0.99226296, 0.00400256])

In [37]:
q_ = model(r, *params)

In [38]:
plt.figure(figsize=(10,10))
plt.plot(r, q)
plt.plot(r, q_)

plt.xlim(rsun-20,rsun+20)
plt.ylim(0,0.7)
plt.tight_layout()

In [13]:
image_ = remove_straylight(image, alpha=alpha, beta=beta, epsilon=epsilon)

In [14]:
plt.figure(figsize=(10,10))
plt.imshow(image_ - Q0, 'inferno', vmin=-0.01, vmax=0.01)

plt.tight_layout()

In [146]:
nx, ny = image.shape
xc, yc, rsun = find_center(image)
xi, yi = np.mgrid[:nx, :ny]

ri = np.sqrt((xi - xc) ** 2 + (yi - yc) ** 2)
r = np.arange(-0.5, np.floor(rsun + 50), 1)

q0 = []
for a, b in zip(r[:-1], r[1:]):
    t = np.where(np.all([ri > a, ri < b], axis=0))
    q0 += [np.nanmedian(image_[t])]

q0 = np.array(q0)
q0 /= np.nanpercentile(q0, 99)
r = (r[:-1] + r[1:]) / 2

In [147]:
plt.figure(figsize=(10,10))
plt.plot(r, q)
plt.plot(r, q0)

plt.xlim(rsun-20,rsun+20)
plt.ylim(0,0.6)
plt.tight_layout()